# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

# Build the Feature Vector

The goal of this feature vector is to represent content performance using signals that would be available before a review decision is made.

All features are observable measurements from search or engagement activity. No target-derived fields or future-window measurements are included.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Target
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

# Feature engineering
df["log_impressions"] = np.log1p(df["impressions_90d"])
df["log_sessions"] = np.log1p(df["sessions_90d"])

# Simple feature vector
feature_cols = [
    "log_impressions",
    "log_sessions",
    "ctr",
    "avg_position",
    "content_age_days"
]

X = df[feature_cols]

# Missing value handling
X = X.fillna(0)

print(X.shape)

X.head()

(30000, 5)


,log_impressions,log_sessions,ctr,avg_position,content_age_days
0,8.243808,2.890372,0.76,10.6,187
1,9.636980,2.302585,0.05,20.3,445
2,9.440023,2.484907,0.09,36.5,141
3,9.371779,4.369448,0.49,6.2,463
4,9.859588,4.983607,0.13,44.0,263


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

# Feature Notes

## log_impressions

Meaning:
Search visibility over the previous 90 days.

Missing values:
Filled with 0.

Available when?
Yes. Impressions are already known before the review decision.

---

## log_sessions

Meaning:
Website sessions during the previous 90 days.

Missing values:
Filled with 0.

Available when?
Yes. Sessions have already occurred.

---

## ctr

Meaning:
Click-through rate from search impressions.

Missing values:
Filled with 0.

Available when?
Yes. Computed from historical clicks and impressions.

---

## avg_position

Meaning:
Average search position.

Missing values:
Filled with 0.

Available when?
Yes. Historical ranking data is available before prediction.

---

## content_age_days

Meaning:
Age of content since publication.

Missing values:
Filled with 0.

Available when?
Yes. Publication date is known when predictions are made.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

# The Leakage Hunt

The purpose of this test is to deliberately introduce leakage and demonstrate how it can create misleading model performance.

A leaky feature directly contains information about the target or uses information that would not be available at prediction time.

The honest model is evaluated first.

Then a deliberately leaky feature is added.

Performance should increase dramatically, demonstrating why leakage detection matters.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

y = df["is_declining_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict_proba(X_test)[:, 1]

honest_auc = roc_auc_score(y_test, preds)

print("Honest ROC AUC:", round(honest_auc,4))

Honest ROC AUC: 0.7362


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

# What I Excluded and Why

## trend_direction

Excluded because it is used to create the target label.

Using it as a feature would directly leak the answer.

---

## is_declining_label

Excluded because it is the prediction target.

Using it as a feature creates perfect leakage.

---

## Any future-window measurements

Excluded because they would not exist at prediction time.

Using future information would make evaluation unrealistic.

---

## Product decision scores or flags

If available, these would be excluded because they represent decisions rather than observable signals.

The goal is to discover signal, not reproduce an existing decision.

---

## Client-identifying fields

Excluded for privacy and because they do not represent content performance behavior.

## Self-check

Before you submit, confirm each line honestly:

- [y ] Every section above is filled — markdown thinking AND the code that backs it
- [y ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ y] No client names, URLs, or private queries anywhere
- [ y] My claims use careful words: observed, measured, directional, decision-support
- [ ]y Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.